# 01 · Generate synthetic data

Twenty-five thousand synthetic members, two toy scoring functions ("Data Science" and "Actuarial"), and **six planted disagreement mechanisms plus one control**. Nothing here is clinical and nothing is trained to predict an outcome — the point is to build a world where I know *why* the scores disagree, then test whether the framework recovers it.

| # | Planted mechanism | How it is injected |
|---|---|---|
| 1 | Missing-data discrepancy | DS built from a raw claims extract; members below 0.80 completeness lose claims |
| 2 | Recency discrepancy | One side scores from an older snapshot and misses claims in the gap |
| 3 | Population-segment discrepancy | DS has diminishing returns + noise that grows with utilization |
| 4 | Transformation/version discrepancy | Staged rollout of DS v2.4, which re-defines `rx_count` |
| 5 | Pipeline/data issue | Duplicate claim lines inflate DS utilization features |
| 6 | Legitimate model disagreement | Clean data, different methodology |
| ctrl | Unexplainable shocks | Idiosyncratic shocks with clean diagnostics — the framework should *not* be able to explain these |

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from src.synthetic import generate
from src import config as C

members, truth = generate(C.N_MEMBERS, C.SEED)
members.head()

,member_id,age_band,region,condition_count,chronic_condition_flag,inpatient_visits,ed_visits,rx_count,months_enrolled,claims_completeness,last_claim_date,claim_lines_total,claim_lines_distinct,gap_claim_load,ds_risk_score,actuarial_risk_score,ds_score_version,act_score_version,ds_snapshot_date,act_snapshot_date
0,M00001,50-59,West,2,1,0,0,1,3,0.980,2025-04-24,5,5,0.0,2.393,2.530,v2.3,a2025.1,2025-12-31,2025-12-31
1,M00002,40-49,West,0,0,0,0,1,12,0.948,2025-06-08,5,5,0.0,0.908,1.169,v2.3,a2025.1,2025-12-31,2025-12-31
2,M00003,60-64,West,1,0,0,1,9,12,0.948,2025-12-28,15,15,0.0,3.601,3.298,v2.3,a2025.1,2025-12-31,2025-12-31
3,M00004,50-59,Southeast,0,0,0,1,2,12,0.984,2025-12-20,5,5,0.0,2.074,1.970,v2.3,a2025.1,2025-12-31,2025-12-31
4,M00005,0-17,Midwest,0,0,0,1,2,12,0.915,2025-03-29,4,4,0.0,1.544,1.224,v2.3,a2025.1,2025-12-31,2025-12-28


## What an analyst sees vs. what only the evaluator sees

`members` is the observable table. `truth` holds each planted mechanism's contribution to Δ and is used **only** to grade the framework — it is never an input to it.

In [2]:
print(members.shape)
members.describe().T[['mean','min','max']].round(2)

(25000, 20)


,mean,min,max
condition_count,0.79,0.00,7.00
chronic_condition_flag,0.19,0.00,1.00
inpatient_visits,0.14,0.00,4.00
ed_visits,0.33,0.00,5.00
rx_count,3.77,0.00,28.00
months_enrolled,10.34,1.00,12.00
claims_completeness,0.93,0.30,1.00
claim_lines_total,9.13,0.00,53.00
claim_lines_distinct,9.03,0.00,50.00
gap_claim_load,0.03,0.00,3.86


In [3]:
truth.planted_primary.value_counts().rename('members (primary planted cause)').to_frame()

,members (primary planted cause)
planted_primary,
METHOD,22221
MODEL_VERSION,982
DATA_COMPLETENESS,792
SNAPSHOT_MISMATCH,665
DUPLICATE_INPUT,205
UNEXPLAINED_SHOCK,135


A note on the planted-cause counts: most members are `METHOD` (ordinary methodology difference, mostly within tolerance). The mechanisms that matter are rare by design, as real defects are.

In [4]:
members.to_csv(ROOT/'data'/'synthetic_members.csv', index=False)
truth.to_csv(ROOT/'data'/'ground_truth_EVAL_ONLY.csv', index=False)
print('saved', len(members), 'members')

saved 25000 members
